# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [8]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue

In [ ]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    metrics = {
        "solution_found": False,
        "nodes_explored": None,
        "states_visited": None,
        "nodes_in_frontier": None,
        "max_depth": None,
        "cost_total": None,
    }
    solution = None

    # Se descartó la heurística de 2^(k-1), donde k es el número de discos fuera de lugar, ya que sobreestima el costo para algunos estados donde los discos ya están ordenados favorablemente. Se optó por h(n) = k, que a pesar de que subestima el costo en algunos casos, es admisible.
    def custom_heuristic(node: NodeHanoi):
        '''
        Heurística personalizada: número de discos fuera de lugar h(n) = k

        La heurística h(n) se define como el número de discos que no están en su posición correcta en el estado actual del nodo n, comparado con el estado objetivo. Cuanto mayor sea el valor de k, más lejos estará el nodo n del estado objetivo. Esta heurística es admisible porque nunca sobreestima el costo real para alcanzar el objetivo, ya que cada disco fuera de lugar requerirá al menos un movimiento para ser colocado correctamente.
        '''
        # k representa el número de discos fuera de lugar. Los discos en la primera y segunda varilla siempre están fuera de lugar, por lo que se suman sus cantidades a k.
        k = len(node.state.rods[0]) + len(node.state.rods[1])
        # Enumera los discos en la tercera varilla y compara con el estado objetivo.
        for index, disk in enumerate(node.state.rods[2]):
            if disk != goal_state.rods[2][index]:
                k += 1
        return k

    # TODO: Completar con el algoritmo de búsqueda que desees implementar
    #####

    # Se inicializa una variable para explorar los nodos
    node = NodeHanoi(initial_state)

    # Se utiliza un set para estados visitados porque ignora duplicados de manera nativa.
    states_visited = set()

    # Se inicializa una variable para contar los nodos explorados.
    nodes_count = 0
    
    # Se inicializa una variable para monitorizar la profundidad máxima alcanzada.
    max_depth = 0

    # Se inicializa una cola de prioridad con f(n) = g(n) + h(n).
    frontier = PriorityQueue(order='min', f=lambda node: node.path_cost + custom_heuristic(node))
    frontier.append(node)

    # Mientras la frontera no esté vacía, se continúa expandiendo nodos.
    while not frontier.heap.empty():
        # Se extrae el nodo que minimiza f(n) = g(n) + h(n). Se descarta el valor de f(n).
        _,node = frontier.pop()
        nodes_count += 1

        # Agregamos el estado a los visitados, si ya estaba no se agrega.
        states_visited.add(node.state)

        # Actualizamos la profundidad máxima alcanzada.
        max_depth = max(max_depth, node.depth)

        # Si el nodo extraído es el objetivo, se retorna la solución y las métricas.
        if problem.goal_test(node.state):
            solution = node
            metrics["solution_found"] = True
            metrics["nodes_explored"] = nodes_count
            metrics["states_visited"] = len(states_visited)
            metrics["nodes_in_frontier"] = len(frontier)
            metrics["max_depth"] = max_depth
            metrics["cost_total"] = node.path_cost
            break

        # Si el nodo extraído no es el objetivo, se expanden sus hijos por cada acción posible.
        for action in problem.actions(node.state):
            child = node.child_node(problem, action)
            # Si el estado del nodo hijo no ha sido visitado, se agrega a la frontera.
            if child.state not in states_visited:
                frontier.append(child)

    return solution, metrics

Se prueba la función:

In [26]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [27]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 329
states_visited: 218
nodes_in_frontier: 37
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [16]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [17]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
